In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Multiply(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return x * y

In [9]:
# Реализуем SE-блок: global average pooling, компактный MLP и масштабирование каналов.
class SEBlock(nn.Module):
    """Squeeze-and-Excitation блок для адаптивного взвешивания каналов."""

    def __init__(self, channels: int, reduction: int = 16) -> None:
        super().__init__()
        hidden_channels = max(channels // reduction, 1)

        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden_channels),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_channels, channels),
            nn.Sigmoid(),
        )

        # Используем вспомогательный модуль для корректного отображения с помощью visualtorch.
        self.multiply = Multiply()

        # Сохраняем последние коэффициенты для последующей визуализации.
        self.last_scale: torch.Tensor | None = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Умножает каждый канал входного тензора на обучаемый коэффициент."""
        batch_size, channels, _, _ = x.shape

        squeezed = self.squeeze(x).view(batch_size, channels)
        scale = self.excitation(squeezed).view(batch_size, channels, 1, 1)

        self.last_scale = scale.detach()

        return self.multiply(x, scale)

In [ ]:
# Каркас ResNet с SE: заполните отмеченные места, используя реализованный ранее BasicBlock как пример.
class SEBasicBlock(nn.Module):
    """Residual-блок с SE-перевзвешиванием перед сложением с shortcut."""

    expansion = 1

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        reduction: int = 16,
    ) -> None:
        super().__init__()

        # Первая свёртка
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # Вторая свёртка
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # SE-блок
        self.se = SEBlock(out_channels, reduction=reduction)

        # Shortcut
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels * self.expansion),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # SE-перевзвешивание перед сложением
        out = self.se(out)

        out += identity
        return self.relu(out)


class ResNetSE(nn.Module):
    """ResNet для CIFAR-10 с SE-блоками в каждом residual-блоке."""

    def __init__(
        self,
        num_classes: int = 10,
        blocks_per_stage: tuple[int, int, int] = (2, 2, 2),
        base_channels: int = 32,
        reduction: int = 16,
    ) -> None:
        super().__init__()

        self.in_channels = base_channels

        self.stem = nn.Sequential(
            nn.Conv2d(3, base_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
        )
        self.stage1 = self._make_stage(
            base_channels, blocks_per_stage[0], stride=1, reduction=reduction
        )
        self.stage2 = self._make_stage(
            base_channels * 2, blocks_per_stage[1], stride=2, reduction=reduction
        )
        self.stage3 = self._make_stage(
            base_channels * 4, blocks_per_stage[2], stride=2, reduction=reduction
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(base_channels * 4, num_classes)

    def _make_stage(
        self, channels: int, blocks: int, stride: int, reduction: int
    ) -> nn.Sequential:
        """Создаёт этап residual-блоков с нужным числом каналов."""
        layers = [
            SEBasicBlock(self.in_channels, channels, stride=stride, reduction=reduction)
        ]
        self.in_channels = channels
        for _ in range(1, blocks):
            layers.append(
                SEBasicBlock(self.in_channels, channels, stride=1, reduction=reduction)
            )
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)